# 使用 LoRA 微调大语言模型（Hugging Face TRL）

## 什么是 LoRA？

LoRA（Low-Rank Adaptation，低秩适配）是一种**参数高效微调（PEFT）**技术。传统全量微调需要更新模型所有参数（成本极高），而 LoRA 的核心思路是：

- **冻结预训练权重**：基础模型参数完全不动，只在旁边"挂"一个轻量 adapter
- **低秩矩阵近似**：对于权重矩阵 W（m×n），学习两个小矩阵 B（m×r）和 A（r×n），用 **ΔW = B×A** 来近似参数更新
- **极少可训练参数**：因为 r ≪ min(m,n)，参数量通常减少 90% 以上
- **效果接近全量微调**：在绝大多数任务上性能损失极小

**关键初始化细节**：训练开始时 B 全部初始化为 0，A 随机初始化（高斯分布）。因此初始 ΔW = B×A = 0，模型行为与原始预训练模型完全一致，不会破坏预训练权重——这是 LoRA 能稳定训练的核心保证。

**推理时**：W_effective = W_frozen + (alpha/r) × B×A，无需改变模型结构。

## 本笔记学习目标

1. 理解 LoRA 各关键参数（rank、alpha、dropout、target_modules）的含义
2. 使用 `trl` 的 `SFTTrainer` + `peft` 的 `LoraConfig` 完成监督微调（SFT）
3. **对比微调前后效果**：训练前保存基础模型回答，训练后用同一问题集做直观对比

## 流程概览

```
加载基础模型 → 记录基础模型回答（基准）→ 配置 LoRA → 配置训练参数
→ 构建 SFTTrainer → 启动训练 → 合并 Adapter（可选）→ 加载微调后模型 → 效果对比
```

## 1. 环境配置

安装必要的 HuggingFace 生态库：

| 库 | 用途 |
|----|------|
| `transformers` | 模型加载、tokenizer、推理 pipeline |
| `datasets` | 数据集加载与处理 |
| `trl` | SFT 训练框架（SFTTrainer） |
| `peft` | LoRA/Adapter 配置与管理 |
| `huggingface_hub` | 模型上传、下载、登录认证 |

In [ ]:
# 安装依赖（Google Colab 中取消注释）
# !pip install transformers datasets trl huggingface_hub peft

# 登录 HuggingFace Hub（用于下载模型/上传结果）
from huggingface_hub import login

login()
# 也可以设置环境变量 HF_TOKEN 来避免每次手动登录

## 2. 加载数据集

本笔记使用 `HuggingFaceTB/smoltalk` 数据集的 `everyday-conversations` 子集，包含 2260 条日常多轮对话。

数据集格式：每条样本包含 `messages` 字段，其中是 `role`（user/assistant）+ `content` 的对话列表。
`SFTTrainer` 会自动将这种格式通过 chat template 转换为训练文本。

In [ ]:
from datasets import load_dataset

# 加载 smoltalk 数据集中的 everyday-conversations 子集
# 该数据集包含日常对话，格式为多轮 messages（role + content）
# 用于监督微调（SFT），教模型以自然对话方式回应用户
dataset = load_dataset(path="HuggingFaceTB/smoltalk", name="everyday-conversations")
dataset

## 3. LoRA 微调训练

LoRA 训练只需三步配置：`LoraConfig`（adapter 结构）→ `SFTConfig`（训练超参数）→ `SFTTrainer`（组装并启动）。

各方案显存需求对比：

| 方案 | 可训练参数比例 | 显存需求 | 典型场景 |
|------|-------------|---------|---------|
| 全量微调（Full FT） | 100% | 极高（需多卡） | 充裕计算资源 |
| LoRA | ~1–5% | 低 | 单卡微调大模型 |
| QLoRA（4bit + LoRA） | ~1–5% | 极低 | 消费级 GPU |

### 3.1 加载基础模型

加载 `SmolLM2-135M` 因果语言模型和对应的 tokenizer，使用 `setup_chat_format` 配置对话格式，为后续 SFT 训练做准备。

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer, setup_chat_format
import torch

# 自动选择最优设备：优先 GPU(CUDA)，其次 Apple Silicon(MPS)，最后 CPU
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"使用设备: {device}")

# 加载基础模型（SmolLM2-135M 是一个小型 135M 参数的因果语言模型）
# 使用小模型便于本地运行，实际生产中可换成更大的模型（如 7B、13B 等）
# 注意：大模型（7B+）需要加 torch_dtype=torch.bfloat16 以节省显存，否则 OOM
model_name = "HuggingFaceTB/SmolLM2-135M"

model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=model_name
).to(device)

tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_name)

# setup_chat_format 做了以下事情：
# 1. 给 tokenizer 添加 chat template（将多轮对话格式化为模型输入文本）
# 2. 添加特殊 token（如 <|im_start|>、<|im_end|>）并调整 model embedding 大小
# 3. 设置 tokenizer.padding_side = "right"（训练时 padding 在右侧）
model, tokenizer = setup_chat_format(model=model, tokenizer=tokenizer)

# 微调后模型的本地保存目录名
finetune_name = "SmolLM2-FT-MyDataset"
# 如果 push_to_hub=True，finetune_tags 会作为 metadata 标签附加到 Hub 仓库
# 当前 push_to_hub=False，此变量仅作记录用途
finetune_tags = ["smol-course", "module_1"]

### 3.2 记录基础模型回答（微调前基准）

在 LoRA 微调开始之前，先用测试问题集跑一遍基础模型并保存回答（`base_results`）。
训练结束后用同一批问题测试微调后的模型，两组回答直接对比，即可看出微调效果。

In [ ]:
from transformers import pipeline

# 定义用于对比测试的提示词（覆盖不同类型：知识问答、代码生成、数学推理、概念辨析）
TEST_PROMPTS = [
    "What is the capital of Germany? Explain why that's the case and if it was different in the past?",
    "Write a Python function to calculate the factorial of a number.",
    "A rectangular garden has a length of 25 feet and a width of 15 feet. If you want to build a fence around the entire garden, how many feet of fencing will you need?",
    "What is the difference between a fruit and a vegetable? Give examples of each.",
]

def run_inference(pipe, prompt, max_new_tokens=200):
    """
    使用 pipeline 对单条 prompt 做推理，返回模型生成的文本。
    apply_chat_template 将 prompt 包装成模型期望的对话格式（如 ChatML 格式），
    add_generation_prompt=True 在末尾添加助手回复的起始标记，引导模型开始生成。
    """
    formatted = pipe.tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )
    outputs = pipe(formatted, max_new_tokens=max_new_tokens)
    # 截掉输入部分，只保留模型新生成的文本
    return outputs[0]["generated_text"][len(formatted):].strip()

# 用 base 模型（微调前）跑一遍测试，把结果保存到 base_results
# 注意：base_results 是一个普通 Python dict，在 kernel 运行期间会一直保留在内存中
# 训练结束后仍可用它与微调后的模型做对比
print("=" * 60)
print("【基础模型（微调前）的回答】")
print("=" * 60)

base_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=device,
)

base_results = {}
for prompt in TEST_PROMPTS:
    response = run_inference(base_pipe, prompt)
    base_results[prompt] = response
    print(f"\n问题: {prompt}")
    print(f"基础模型回答: {response}")
    print("-" * 60)

# ── 推理完成后立即释放 base_pipe，避免训练阶段同时占用两份 GPU 显存 ──
del base_pipe
torch.cuda.empty_cache()
print("\n基础模型推理完成，显存已释放，准备开始 LoRA 训练...")

### 3.3 配置 LoRA 参数（LoraConfig）

定义低秩适配矩阵的关键超参数，决定 adapter 的表达能力与参数效率。代码注释中包含每个参数的详细含义。

In [ ]:
from peft import LoraConfig

# ──────────────────────────────────────────────────────────────
# LoRA 原理简述：
# 对于权重矩阵 W（形状 m×n），LoRA 不直接更新 W，而是学习两个小矩阵：
#   B（m×r）初始化为 0；A（r×n）随机初始化（高斯）
# 训练开始时 ΔW = B×A = 0，与原始模型等价，训练过程中逐步学习任务相关偏移量
# 推理时：W_effective = W_frozen + (alpha/r) × B×A
# 参数量：m×n → r×(m+n)，节省约 90%+（以 r=8, m=n=4096 为例：节省 99.6%）
# ──────────────────────────────────────────────────────────────

# r（rank，秩）：LoRA 矩阵的秩维度
# - 决定了低秩矩阵 B、A 的中间维度大小
# - 越大 → 表达能力越强，但参数量和显存占用也越多
# - 越小 → 压缩比越高，适合简单任务或资源受限场景
# - 典型范围：4（极致压缩）~ 64（高表达力）；通常用 8 或 16
rank_dimension = 8

# lora_alpha（缩放因子）：控制 LoRA 更新的强度
# - 实际缩放公式：scale = lora_alpha / r
# - 相当于调节"LoRA 对原始权重影响力"的旋钮
# - 常见设置：等于 r（scale=1，更新幅度适中）或 2×r（scale=2，更强的适应力）
# - 这里 alpha=8, r=8 → scale=1，LoRA 更新与基础权重等比例贡献
lora_alpha = 8

# lora_dropout：LoRA 层的 dropout 概率
# - 在训练时随机将一部分 LoRA 激活置零，起到正则化作用
# - 防止 adapter 过拟合到训练集
# - 数据量充足（>10k 样本）时可设为 0.0；数据较少时建议 0.05~0.1
lora_dropout = 0.05

peft_config = LoraConfig(
    r=rank_dimension,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    # bias：是否也训练 bias 参数
    # "none" → 只训练 LoRA 矩阵，bias 完全冻结（最节省参数，最常用）
    # "all" → 所有 bias 都可训练
    # "lora_only" → 只训练被 LoRA 应用到的层的 bias
    bias="none",
    # target_modules：指定对哪些子模块应用 LoRA
    # "all-linear" → 对所有线性层（Q、K、V、O、FFN 的 up/down/gate 等）都加 LoRA
    # 也可以明确指定层名，如 ["q_proj", "v_proj"]（参数更少，效果略差）
    # 不同模型的层名不同，可用 model.named_modules() 查看
    target_modules="all-linear",
    # task_type：告知 PEFT 这是因果语言模型（自回归生成任务）
    # 影响 PEFT 内部如何处理模型输出和 loss
    task_type="CAUSAL_LM",
)

### 3.4 配置训练超参数（SFTConfig）

配置学习率、批次大小、序列长度、packing、优化器、精度等训练参数。代码注释中包含每个参数的详细说明和选取依据。

> **注意**：在 TRL 新版本中，`packing`、`max_seq_length`、`dataset_kwargs` 已从 `SFTTrainer` 迁移到 `SFTConfig`，直接传给 `SFTTrainer` 会报 `TypeError`。

In [ ]:
# SFTConfig 继承自 HuggingFace TrainingArguments，专为 SFT 场景设计
args = SFTConfig(
    # ── 输出 ──────────────────────────────────────────────────
    # 模型 checkpoint 保存到本地哪个目录
    output_dir=finetune_name,

    # ── 训练轮数 ───────────────────────────────────────────────
    # 遍历完整训练集的次数；1 轮通常用于快速验证，正式训练可设 3~5
    num_train_epochs=1,

    # ── 批次大小 ───────────────────────────────────────────────
    # 每张 GPU 每步处理的样本数；受显存限制，通常设为 1~4
    per_device_train_batch_size=2,
    # 梯度累积步数：每 N 步才做一次参数更新
    # 等效 batch size = per_device_train_batch_size × gradient_accumulation_steps × GPU 数量
    # 这里等效 batch = 2 × 2 = 4，用小显存模拟大 batch 的训练效果
    gradient_accumulation_steps=2,

    # ── 序列长度与 Packing ──────────────────────────────────────
    # max_seq_length：截断/拼接序列的最大长度（token 数）
    # 超过此长度的样本会被截断；packing 时多个短样本会被拼接以填满此长度
    max_seq_length=1512,
    # packing：将多条短样本拼接填满 max_seq_length，避免大量 padding
    # 对话数据集中样本通常较短，packing 可将 GPU 利用率从约 30% 提升到 90%+
    packing=True,
    # dataset_kwargs：数据集预处理参数（packing 时生效）
    dataset_kwargs={
        # add_special_tokens=False：chat_template 已处理特殊 token，无需重复添加
        "add_special_tokens": False,
        # append_concat_token=False：packing 时不在样本间插入额外分隔符
        "append_concat_token": False,
    },

    # ── 显存优化 ───────────────────────────────────────────────
    # gradient_checkpointing：以重计算换显存
    # 正向传播时不保存中间激活值，反向传播时重新计算；节省约 30-40% 显存，但速度变慢
    gradient_checkpointing=True,

    # ── 优化器 ────────────────────────────────────────────────
    # adamw_torch_fused：PyTorch >= 2.0 的融合版 AdamW，将多个 CUDA kernel 合并执行
    # 相比普通 AdamW 快约 10-15%，且精度相同（如报错，可改为 "adamw_torch"）
    optim="adamw_torch_fused",
    # 学习率：LoRA 微调推荐值比全量微调大（因为只更新少量参数，影响范围有限）
    learning_rate=2e-4,
    # 梯度裁剪阈值：防止梯度爆炸；将梯度范数限制在 0.3 以内
    max_grad_norm=0.3,

    # ── 学习率调度 ─────────────────────────────────────────────
    # warmup_ratio：热身阶段占总步数的比例
    # 前 3% 的步骤学习率从 0 线性增长到 learning_rate，避免初始大梯度破坏预训练权重
    warmup_ratio=0.03,
    # 热身结束后保持学习率不变（适合短时间微调；长时间训练可改为 "cosine"）
    lr_scheduler_type="constant",

    # ── 日志与保存 ─────────────────────────────────────────────
    logging_steps=10,       # 每 10 步打印一次 loss 等指标
    save_strategy="epoch",  # 每个 epoch 结束时保存 checkpoint

    # ── 精度 ──────────────────────────────────────────────────
    # bf16：bfloat16 混合精度训练，相比 fp32 显存减半、速度加倍
    # 需要 Ampere 架构 GPU（A10G/A100/RTX 3090+）；T4/V100 不支持，会自动回退到 fp32
    bf16=torch.cuda.is_bf16_supported(),

    # ── 其他 ──────────────────────────────────────────────────
    push_to_hub=False,   # 不自动上传到 HuggingFace Hub
    report_to="none",    # 不接入 W&B / TensorBoard 等实验追踪平台
)

### 3.5 构建训练器（SFTTrainer）

将模型、数据集、LoRA 配置和训练参数组合为 `SFTTrainer`。传入 `peft_config` 后，训练器会自动冻结基础模型并注入 LoRA 矩阵。序列长度、packing 等参数已在 `SFTConfig` 中统一配置，这里只需传入核心组件。最后打印可训练参数量，直观感受 LoRA 的参数效率。

In [ ]:
# SFTTrainer 在 Trainer 基础上增加了以下能力：
# 1. 自动将数据集 messages 字段通过 chat_template 格式化为训练文本
# 2. 原生集成 PEFT（只需传入 peft_config，剩下的自动处理）
# 3. 支持 packing、max_seq_length 等（已在 SFTConfig/args 中统一配置）
trainer = SFTTrainer(
    model=model,
    args=args,                        # 包含所有训练超参数（含 packing、max_seq_length、dataset_kwargs）
    train_dataset=dataset["train"],
    # 传入 LoRA 配置后，SFTTrainer 会自动：
    # - 冻结 base model 所有参数（requires_grad=False）
    # - 在 target_modules 上注入可训练的 LoRA 矩阵（B 初始化为 0）
    # - 训练时只更新这些 LoRA 参数
    peft_config=peft_config,
    tokenizer=tokenizer,
)

# 打印可训练参数量，直观感受 LoRA 的参数效率
# LoRA 正确注入后，可训练比例通常在 0.5%~5% 之间
# 如果显示接近 100%，说明 peft_config 未生效，需检查配置
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"可训练参数: {trainable:,} / 总参数: {total:,} ({100 * trainable / total:.2f}%)")

### 3.6 启动训练

调用 `trainer.train()` 开始训练。由于使用 PEFT，训练结束后 `save_model()` 只保存 adapter 权重（通常几 MB），而非完整模型（几 GB）。

In [ ]:
# 启动训练
trainer.train()

# 保存 adapter 权重到 output_dir（即 finetune_name 目录）
# 由于使用 PEFT，save_model() 只保存 adapter 文件（几 MB），而非完整模型（几 GB）
# 保存内容：adapter_config.json、adapter_model.safetensors、tokenizer_config.json 等
# Section 5 将从此目录重新加载微调后的模型
trainer.save_model()

> **训练耗时参考**：本笔记使用 SmolLM2-135M（135M 参数）+ 2260 条对话训练 1 epoch，在消费级 GPU（如 T4）上约需 **2~5 分钟**。如需更好效果，可将 `num_train_epochs` 改为 3（约 10~15 分钟）。在 A10G（`g5.2xlarge`，$1.21/h）上更快，整体成本极低。

## 4. （可选）合并 LoRA Adapter

训练完成后，adapter 权重（仅几 MB）与基础模型是分离存储的。根据使用场景选择部署方式：

| 部署方式 | 优点 | 缺点 | 适用场景 |
|---------|------|------|---------|
| **分离保存**（adapter + base） | 可随时切换多个 adapter、可继续训练 | 推理时有额外计算开销 | 多任务、科研实验 |
| **合并后部署**（`merge_and_unload`） | 推理更快、部署简单、框架兼容性好 | 无法再拆分 adapter | 生产上线 |

`merge_and_unload()` 将 ΔW = B×A 的计算结果加回原始权重 W，返回一个普通的 `transformers` 模型。

> **注意**：合并后的模型保存到独立目录（`finetune_name + "-merged"`），**不会覆盖** adapter 目录（`finetune_name`），Section 5 仍可正常从 adapter 目录加载进行效果对比。

In [ ]:
from peft import AutoPeftModelForCausalLM

# 用独立变量名加载，避免覆盖训练时的 model 变量
# low_cpu_mem_usage=True：在 CPU 上逐层加载，减少峰值内存
peft_model_to_merge = AutoPeftModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=args.output_dir,  # adapter 目录
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

# merge_and_unload()：将 ΔW = B×A 加回冻结的 W，返回普通 transformers 模型
# 合并后 adapter 矩阵消失，模型结构与原始基础模型完全相同，可直接用 AutoModelForCausalLM 加载
merged_model = peft_model_to_merge.merge_and_unload()

# 释放 PEFT 模型占用的内存，只保留合并后的模型
del peft_model_to_merge

# 保存到独立目录，不覆盖 adapter 目录（Section 5 仍需要从 adapter 目录加载）
merged_save_dir = finetune_name + "-merged"
merged_model.save_pretrained(merged_save_dir, safe_serialization=True, max_shard_size="2GB")
# tokenizer 与模型权重强耦合（embedding 大小必须匹配），必须一起保存
tokenizer.save_pretrained(merged_save_dir)

print(f"合并后模型已保存到本地目录: {merged_save_dir}/")
print("可用 AutoModelForCausalLM.from_pretrained(merged_save_dir) 直接加载")

## 5. 效果对比：微调前 vs 微调后

**对比方式说明**：
- `base_results`：在 Section 3.2 中用基础模型（训练前）跑的回答，存储在内存中
- `ft_results`：用从本地 adapter 目录重新加载的微调后模型跑的回答

两者使用完全相同的测试问题和推理函数（`run_inference`），直接对比输出即可看出微调效果。

> **关于 `push_to_hub=False`**：模型保存在本地目录 `finetune_name/`（即 `SmolLM2-FT-MyDataset/`），`AutoPeftModelForCausalLM.from_pretrained()` 会优先查找本地路径，无需从 Hub 下载。

### 5.1 加载微调后模型

释放训练阶段的显存，加载保存的 LoRA adapter，构建用于推理的 pipeline。

In [ ]:
# 释放训练阶段占用的所有显存，避免加载 ft_model 时 OOM
del trainer
del model        # 训练时的 LoRA 模型（含 adapter 权重，已 save_model() 到磁盘）

# 如果运行了 Section 4（可选合并），merged_model 也可能还在内存中
try:
    del merged_model
except NameError:
    pass

torch.cuda.empty_cache()
print("显存已释放，准备加载微调后模型...")

In [ ]:
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer, pipeline

# 从本地 adapter 目录加载微调后的模型
# finetune_name 目录结构：adapter_config.json + adapter_model.safetensors + tokenizer 文件
# AutoPeftModelForCausalLM 会先从 HF 缓存加载基础模型，再挂载 LoRA adapter
tokenizer = AutoTokenizer.from_pretrained(finetune_name)
ft_model = AutoPeftModelForCausalLM.from_pretrained(
    finetune_name,
    device_map="auto",      # 自动分配到可用 GPU/CPU
    torch_dtype=torch.float16,
)

# 构建微调后模型的推理 pipeline
# 注意：model 已通过 device_map="auto" 放置到设备，pipeline 无需再指定 device
ft_pipe = pipeline(
    "text-generation",
    model=ft_model,
    tokenizer=tokenizer,
)
print("微调后模型加载完成，开始对比推理...")

### 5.2 输出对比结果

用微调后的模型跑同一批测试问题，与 Section 3.2 保存的 `base_results` 对比。重点观察：
- 微调后是否更好地遵循了对话格式（assistant 角色风格）
- 指令跟随能力是否有提升（直接回答 vs 重复 prompt/无意义续写）
- 回复风格是否向 everyday-conversations 数据集的风格靠拢（简洁口语化）

In [ ]:
# 防御性检查：base_results 在 Section 3.2 中产生，若重启 kernel 后直接运行此 cell 会报错
assert "base_results" in dir(), (
    "base_results 未定义！请先运行 Section 3.2（cell 9）跑基础模型推理，再运行此 cell。"
)

print("=" * 60)
print("【微调前 vs 微调后 回答对比】")
print("=" * 60)

ft_results = {}
for prompt in TEST_PROMPTS:
    ft_response = run_inference(ft_pipe, prompt)
    ft_results[prompt] = ft_response

    print(f"\n{'='*60}")
    print(f"问题: {prompt}")
    print(f"\n【微调前（base model）】\n{base_results[prompt]}")
    print(f"\n【微调后（LoRA adapter）】\n{ft_response}")

print("\n" + "=" * 60)
print("【观察要点】")
print("""
微调前后的主要差异通常体现在：

1. 回复格式：
   - 微调前：模型可能不遵循对话格式，输出杂乱或截断
   - 微调后：遵循 assistant 角色，给出结构清晰的回复

2. 话题相关性（针对 everyday-conversations 数据集）：
   - 微调后的模型更擅长日常对话风格的表达
   - 回复更简洁、口语化，而非生硬的知识堆砌

3. 指令跟随能力：
   - 微调后能更准确地理解并回应用户问题
   - 避免重复 prompt 内容或产生无意义续写

如果效果差异不明显，可尝试：
  → num_train_epochs=3（更多训练轮次）
  → rank_dimension=16（更强的 adapter 表达力）
  → 换用更大的基础模型（如 SmolLM2-1.7B 或 Llama 7B）
""")

## 6. 各 GPU 配置训练时间估算

### 当前训练规模

| 维度 | 配置 |
|------|------|
| 模型 | SmolLM2-135M（1.35 亿参数） |
| LoRA | r=8，target_modules="all-linear" |
| 数据集 | smoltalk everyday-conversations，2260 条对话 |
| 序列处理 | packing=True，max_seq_length=1512 |
| 有效训练步数 | 约 150 步（2260 条 → packing 后约 300 个序列，per_device_batch_size=2，1 epoch） |
| 精度 | bf16（Ampere 架构及以上支持） |

---

### 各阶段耗时对比

| 阶段 | T4（参考基线） | RTX 5090 32G × 1 | RTX 5090 32G × 4 | H100 SXM 80G | H200 SXM 141G |
|------|------------|-----------------|-----------------|-------------|--------------|
| 数据集 tokenize/packing（CPU） | ~20s | ~20s | ~20s | ~20s | ~20s |
| 模型加载 | ~5s | ~5s | ~5s | ~5s | ~5s |
| 基础模型推理（4 条测试） | ~5s | ~2s | ~2s | ~2s | ~2s |
| **LoRA 训练（1 epoch）** | **2–3 分钟** | **~20–40s** | **~8–15s** | **~15–25s** | **~12–20s** |
| 保存 / 合并 / ft 推理 | ~10s | ~5s | ~5s | ~5s | ~5s |
| **全流程合计** | **3–5 分钟** | **~1–1.5 分钟** | **~1 分钟** | **~1–1.5 分钟** | **~1 分钟以内** |

---

### 关键说明

**5090 × 4 卡并不比单卡快太多**

SmolLM2-135M 计算量极小，DDP 的 AllReduce 梯度同步通信开销会占据相当比例。实际加速比约为 2–2.5x，而非理论 4x。

**H100 SXM vs RTX 5090**

H100 SXM 峰值 BF16 算力（~989 TFLOPS dense）略高于 5090（~600–700 TFLOPS dense），但 135M 模型计算量极小，两者实际训练时间非常接近。H100 的优势主要体现在大 batch / 大模型场景。

**H200 vs H100**

H200 与 H100 SXM 计算核心相同，主要升级是内存容量（80G HBM3 → 141G HBM3e）和带宽（3.35 TB/s → 4.8 TB/s）。对 135M 模型的训练速度几乎无差别；内存带宽优势在大 batch 推理时才显现。

**总结**

本任务（135M 模型 + 2260 条数据 + 1 epoch）在任何现代 GPU 上均为分钟级别。若要体现不同 GPU 的性能差距，需换用 **7B+ 模型**或更大规模数据集。